# Modelo multimodal para velocidad de adopción en PetFinder

Este notebook construye un enfoque integral con:

1. EDA
2. Feature engineering
3. Modelos tabulares
4. Modelos de texto
5. Modelo de imágenes
6. Ensambles generales y ensambles condicionados por velocidad de adopción

Importante: en el dataset original `AdoptionSpeed` toma valores `0, 1, 2, 3, 4`.
Si querés pensarlo como `1..5`, podés sumar 1 solo en reportes o visualizaciones.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFilter, ImageOps

from scipy.stats import entropy

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
BASE_DIR = Path(r'C:\Maestria\Laboratorio de Implementacion 2')
TRAIN_CSV = BASE_DIR / 'train' / 'train.csv'
TRAIN_IMAGES_DIR = BASE_DIR / 'train_images'
TRAIN_METADATA_DIR = BASE_DIR / 'train_metadata'
TRAIN_SENTIMENT_DIR = BASE_DIR / 'train_sentiment'
CACHE_DIR = BASE_DIR / 'cache_multimodal'
CACHE_DIR.mkdir(exist_ok=True)

for path in [TRAIN_CSV, TRAIN_IMAGES_DIR, TRAIN_METADATA_DIR, TRAIN_SENTIMENT_DIR, CACHE_DIR]:
    print(f'{path.name}:', 'OK' if path.exists() else 'NO ENCONTRADO')

## Carga y EDA

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df['image_path'] = df['PetID'].map(lambda pet_id: TRAIN_IMAGES_DIR / f'{pet_id}-1.jpg')
df['has_main_image'] = df['image_path'].map(Path.exists)

print('Shape:', df.shape)
display(df.head())
display(df['AdoptionSpeed'].value_counts().sort_index().rename('count').to_frame())

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)
display(missing[missing['missing_count'] > 0])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=df, x='AdoptionSpeed', ax=axes[0, 0])
axes[0, 0].set_title('Distribución de AdoptionSpeed (0..4)')

sns.boxplot(data=df, x='AdoptionSpeed', y='Age', ax=axes[0, 1])
axes[0, 1].set_title('Edad por AdoptionSpeed')

sns.boxplot(data=df, x='AdoptionSpeed', y='Fee', ax=axes[1, 0])
axes[1, 0].set_title('Fee por AdoptionSpeed')

sns.boxplot(data=df, x='AdoptionSpeed', y='PhotoAmt', ax=axes[1, 1])
axes[1, 1].set_title('Cantidad de fotos por AdoptionSpeed')

plt.tight_layout()

## Feature engineering

Vamos a construir:

- features tabulares derivadas
- features de texto desde `Description` y `Name`
- features de imagen desde la primera foto
- features simples desde `sentiment` y `metadata` cuando estén disponibles

In [ ]:
def safe_text(x):
    if pd.isna(x):
        return ''
    return str(x)

def word_count(text):
    return len(safe_text(text).split())

def char_count(text):
    return len(safe_text(text))

def uppercase_ratio(text):
    text = safe_text(text)
    if not text:
        return 0.0
    alpha = [c for c in text if c.isalpha()]
    if not alpha:
        return 0.0
    return sum(c.isupper() for c in alpha) / len(alpha)

def exclamation_count(text):
    return safe_text(text).count('!')

def load_sentiment_features(pet_id):
    path = TRAIN_SENTIMENT_DIR / f'{pet_id}.json'
    if not path.exists():
        return {'sent_mag': np.nan, 'sent_score': np.nan}
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        score = data.get('documentSentiment', {}).get('score', np.nan)
        mag = data.get('documentSentiment', {}).get('magnitude', np.nan)
        return {'sent_mag': mag, 'sent_score': score}
    except Exception:
        return {'sent_mag': np.nan, 'sent_score': np.nan}

def load_metadata_features(pet_id):
    path = TRAIN_METADATA_DIR / f'{pet_id}-1.json'
    if not path.exists():
        return {
            'img_crop_conf': np.nan,
            'img_dominant_entropy': np.nan,
            'img_label_score_mean': np.nan,
            'img_label_count': 0
        }
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        crop_conf = data.get('cropHintsAnnotation', {}).get('cropHints', [{}])[0].get('confidence', np.nan)
        colors = data.get('imagePropertiesAnnotation', {}).get('dominantColors', {}).get('colors', [])
        color_scores = [c.get('score', 0.0) for c in colors]
        color_entropy = entropy(color_scores) if len(color_scores) > 1 else 0.0
        labels = data.get('labelAnnotations', [])
        label_scores = [x.get('score', 0.0) for x in labels]
        return {
            'img_crop_conf': crop_conf,
            'img_dominant_entropy': color_entropy,
            'img_label_score_mean': float(np.mean(label_scores)) if label_scores else 0.0,
            'img_label_count': len(label_scores)
        }
    except Exception:
        return {
            'img_crop_conf': np.nan,
            'img_dominant_entropy': np.nan,
            'img_label_score_mean': np.nan,
            'img_label_count': 0
        }

def handcrafted_image_features(image_path, image_size=(64, 64), bins=16):
    if not Path(image_path).exists():
        return {
            'img_mean_r': np.nan, 'img_mean_g': np.nan, 'img_mean_b': np.nan,
            'img_std_r': np.nan, 'img_std_g': np.nan, 'img_std_b': np.nan,
            'img_gray_mean': np.nan, 'img_gray_std': np.nan,
            'img_edge_mean': np.nan, 'img_edge_std': np.nan,
            'img_brightness_q25': np.nan, 'img_brightness_q75': np.nan
        }

    img = Image.open(image_path).convert('RGB').resize(image_size)
    rgb = np.asarray(img, dtype=np.float32) / 255.0
    gray_img = ImageOps.grayscale(img)
    gray = np.asarray(gray_img, dtype=np.float32) / 255.0
    edges = np.asarray(gray_img.filter(ImageFilter.FIND_EDGES), dtype=np.float32) / 255.0

    return {
        'img_mean_r': rgb[:, :, 0].mean(),
        'img_mean_g': rgb[:, :, 1].mean(),
        'img_mean_b': rgb[:, :, 2].mean(),
        'img_std_r': rgb[:, :, 0].std(),
        'img_std_g': rgb[:, :, 1].std(),
        'img_std_b': rgb[:, :, 2].std(),
        'img_gray_mean': gray.mean(),
        'img_gray_std': gray.std(),
        'img_edge_mean': edges.mean(),
        'img_edge_std': edges.std(),
        'img_brightness_q25': np.percentile(gray, 25),
        'img_brightness_q75': np.percentile(gray, 75)
    }

In [ ]:
feature_df = df.copy()

feature_df['Name'] = feature_df['Name'].fillna('Unknown')
feature_df['Description'] = feature_df['Description'].fillna('')
feature_df['name_word_count'] = feature_df['Name'].map(word_count)
feature_df['desc_word_count'] = feature_df['Description'].map(word_count)
feature_df['desc_char_count'] = feature_df['Description'].map(char_count)
feature_df['desc_upper_ratio'] = feature_df['Description'].map(uppercase_ratio)
feature_df['desc_exclamation_count'] = feature_df['Description'].map(exclamation_count)
feature_df['has_name'] = (feature_df['Name'].str.strip().str.lower() != 'no name yet').astype(int)
feature_df['has_description'] = (feature_df['Description'].str.strip() != '').astype(int)
feature_df['fee_per_photo'] = feature_df['Fee'] / (feature_df['PhotoAmt'].fillna(0) + 1)
feature_df['age_per_quantity'] = feature_df['Age'] / feature_df['Quantity'].replace(0, 1)
feature_df['is_mixed_breed'] = (feature_df['Breed2'] != 0).astype(int)
feature_df['is_mixed_color'] = ((feature_df['Color2'] != 0) | (feature_df['Color3'] != 0)).astype(int)
feature_df['media_richness'] = feature_df['PhotoAmt'].fillna(0) + 2 * feature_df['VideoAmt'].fillna(0)

sentiment_rows = [load_sentiment_features(pet_id) for pet_id in feature_df['PetID']]
metadata_rows = [load_metadata_features(pet_id) for pet_id in feature_df['PetID']]
image_rows = [handcrafted_image_features(path) for path in feature_df['image_path']]

feature_df = pd.concat([
    feature_df,
    pd.DataFrame(sentiment_rows),
    pd.DataFrame(metadata_rows),
    pd.DataFrame(image_rows)
], axis=1)

feature_df['text_full'] = (feature_df['Name'].fillna('') + ' ' + feature_df['Description'].fillna('')).str.strip()

print('Shape luego del feature engineering:', feature_df.shape)
display(feature_df.head())

## Split

Usamos un split único `train / valid / test` para comparar de forma consistente todos los modelos.

In [ ]:
train_df, temp_df = train_test_split(
    feature_df,
    test_size=0.30,
    random_state=SEED,
    stratify=feature_df['AdoptionSpeed']
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['AdoptionSpeed']
)

print(len(train_df), len(valid_df), len(test_df))

## Modelo tabular

In [ ]:
target_col = 'AdoptionSpeed'

categorical_cols = [
    'Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
    'Health', 'State', 'RescuerID'
]

numeric_cols = [
    'Age', 'Quantity', 'Fee', 'VideoAmt', 'PhotoAmt',
    'name_word_count', 'desc_word_count', 'desc_char_count', 'desc_upper_ratio',
    'desc_exclamation_count', 'has_name', 'has_description', 'fee_per_photo',
    'age_per_quantity', 'is_mixed_breed', 'is_mixed_color', 'media_richness',
    'sent_mag', 'sent_score', 'img_crop_conf', 'img_dominant_entropy',
    'img_label_score_mean', 'img_label_count', 'img_mean_r', 'img_mean_g', 'img_mean_b',
    'img_std_r', 'img_std_g', 'img_std_b', 'img_gray_mean', 'img_gray_std',
    'img_edge_mean', 'img_edge_std', 'img_brightness_q25', 'img_brightness_q75'
]

tab_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

tab_model_lr = Pipeline([
    ('prep', tab_preprocessor),
    ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', multi_class='multinomial', random_state=SEED))
])

tab_model_lgbm = Pipeline([
    ('prep', tab_preprocessor),
    ('clf', LGBMClassifier(
        objective='multiclass',
        num_class=5,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        verbose=-1
    ))
])

tab_model_xgb = Pipeline([
    ('prep', tab_preprocessor),
    ('clf', XGBClassifier(
        objective='multi:softprob',
        num_class=5,
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='mlogloss',
        random_state=SEED
    ))
])

## Modelo de texto

In [ ]:
text_model_tfidf = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        ngram_range=(1, 2),
        min_df=3,
        max_features=40000,
        sublinear_tf=True
    )),
    ('svd', TruncatedSVD(n_components=300, random_state=SEED)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', multi_class='multinomial', random_state=SEED))
])

## Modelo de imágenes

Acá usamos las features visuales manuales construidas arriba como baseline visual estable en este entorno.

In [ ]:
image_feature_cols = [
    'img_crop_conf', 'img_dominant_entropy', 'img_label_score_mean', 'img_label_count',
    'img_mean_r', 'img_mean_g', 'img_mean_b', 'img_std_r', 'img_std_g', 'img_std_b',
    'img_gray_mean', 'img_gray_std', 'img_edge_mean', 'img_edge_std',
    'img_brightness_q25', 'img_brightness_q75'
]

image_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', multi_class='multinomial', random_state=SEED))
])

## Entrenamiento y evaluación base

In [ ]:
def evaluate_predictions(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic')
    }

results = []
fitted_models = {}
valid_probas = {}
test_probas = {}

model_specs = {
    'tab_lr': {
        'model': tab_model_lr,
        'X_train': train_df[categorical_cols + numeric_cols],
        'X_valid': valid_df[categorical_cols + numeric_cols],
        'X_test': test_df[categorical_cols + numeric_cols]
    },
    'tab_lgbm': {
        'model': tab_model_lgbm,
        'X_train': train_df[categorical_cols + numeric_cols],
        'X_valid': valid_df[categorical_cols + numeric_cols],
        'X_test': test_df[categorical_cols + numeric_cols]
    },
    'tab_xgb': {
        'model': tab_model_xgb,
        'X_train': train_df[categorical_cols + numeric_cols],
        'X_valid': valid_df[categorical_cols + numeric_cols],
        'X_test': test_df[categorical_cols + numeric_cols]
    },
    'text_tfidf': {
        'model': text_model_tfidf,
        'X_train': train_df['text_full'],
        'X_valid': valid_df['text_full'],
        'X_test': test_df['text_full']
    },
    'image_lr': {
        'model': image_model,
        'X_train': train_df[image_feature_cols],
        'X_valid': valid_df[image_feature_cols],
        'X_test': test_df[image_feature_cols]
    }
}

y_train = train_df[target_col]
y_valid = valid_df[target_col]
y_test = test_df[target_col]

for model_name, spec in model_specs.items():
    fitted = clone(spec['model'])
    fitted.fit(spec['X_train'], y_train)
    pred_valid = fitted.predict(spec['X_valid'])
    prob_valid = fitted.predict_proba(spec['X_valid'])
    prob_test = fitted.predict_proba(spec['X_test'])
    metrics = evaluate_predictions(y_valid, pred_valid)
    results.append({'model': model_name, **metrics})
    fitted_models[model_name] = fitted
    valid_probas[model_name] = prob_valid
    test_probas[model_name] = prob_test

results_df = pd.DataFrame(results).sort_values(['qwk', 'accuracy'], ascending=False).reset_index(drop=True)
display(results_df)

## Ensambles combinados

Primero hacemos un blend general. Después construimos un blend condicionado para favorecer mejor la adopción rápida o tardía.

In [ ]:
ensemble_weights_general = {
    'tab_lgbm': 0.45,
    'tab_xgb': 0.20,
    'text_tfidf': 0.20,
    'image_lr': 0.15
}

def blend_probabilities(proba_dict, weights):
    total = None
    weight_sum = 0.0
    for model_name, weight in weights.items():
        if total is None:
            total = weight * proba_dict[model_name]
        else:
            total += weight * proba_dict[model_name]
        weight_sum += weight
    return total / weight_sum

valid_blend_general = blend_probabilities(valid_probas, ensemble_weights_general)
pred_valid_blend_general = valid_blend_general.argmax(axis=1)
general_metrics = evaluate_predictions(y_valid, pred_valid_blend_general)
general_metrics

In [ ]:
fast_classes = [0, 1]
late_classes = [3, 4]

ensemble_weights_fast = {
    'image_lr': 0.35,
    'text_tfidf': 0.25,
    'tab_lgbm': 0.25,
    'tab_xgb': 0.15
}

ensemble_weights_late = {
    'tab_lgbm': 0.45,
    'tab_xgb': 0.25,
    'text_tfidf': 0.20,
    'image_lr': 0.10
}

ensemble_weights_middle = {
    'tab_lgbm': 0.40,
    'text_tfidf': 0.25,
    'tab_xgb': 0.20,
    'image_lr': 0.15
}

def region_aware_blend(proba_dict, base_weights, fast_weights, middle_weights, late_weights, fast_thr=0.50, late_thr=0.50):
    base = blend_probabilities(proba_dict, base_weights)
    fast = blend_probabilities(proba_dict, fast_weights)
    middle = blend_probabilities(proba_dict, middle_weights)
    late = blend_probabilities(proba_dict, late_weights)

    out = np.zeros_like(base)
    for i in range(len(base)):
        fast_mass = base[i, fast_classes].sum()
        late_mass = base[i, late_classes].sum()

        if fast_mass >= fast_thr:
            out[i] = fast[i]
        elif late_mass >= late_thr:
            out[i] = late[i]
        else:
            out[i] = middle[i]
    return out

valid_blend_region = region_aware_blend(
    valid_probas,
    ensemble_weights_general,
    ensemble_weights_fast,
    ensemble_weights_middle,
    ensemble_weights_late,
    fast_thr=0.52,
    late_thr=0.52
)

pred_valid_blend_region = valid_blend_region.argmax(axis=1)
region_metrics = evaluate_predictions(y_valid, pred_valid_blend_region)
region_metrics

## Comparación sobre validación

In [ ]:
extra_rows = pd.DataFrame([
    {'model': 'blend_general', **general_metrics},
    {'model': 'blend_region_aware', **region_metrics}
])
all_results_df = pd.concat([results_df, extra_rows], ignore_index=True).sort_values(['qwk', 'accuracy'], ascending=False).reset_index(drop=True)
display(all_results_df)

## Evaluación final en test

In [ ]:
best_model_name = all_results_df.iloc[0]['model']

if best_model_name in test_probas:
    test_pred = test_probas[best_model_name].argmax(axis=1)
elif best_model_name == 'blend_general':
    test_blend = blend_probabilities(test_probas, ensemble_weights_general)
    test_pred = test_blend.argmax(axis=1)
elif best_model_name == 'blend_region_aware':
    test_blend = region_aware_blend(
        test_probas,
        ensemble_weights_general,
        ensemble_weights_fast,
        ensemble_weights_middle,
        ensemble_weights_late,
        fast_thr=0.52,
        late_thr=0.52
    )
    test_pred = test_blend.argmax(axis=1)
else:
    raise ValueError(f'Modelo no reconocido: {best_model_name}')

final_metrics = evaluate_predictions(y_test, test_pred)
print('Mejor modelo en validación:', best_model_name)
print(final_metrics)
print(classification_report(y_test, test_pred))

In [ ]:
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de confusión - {best_model_name}')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.tight_layout()

## Sugerencia de estrategia

La idea de ponderar más imágenes para adopción rápida y más tabular para adopción tardía me parece buena.

Hipótesis razonable:

- adopción rápida (`0` o `1`): la foto, el impacto visual y el texto corto/emocional pesan bastante
- adopción intermedia (`2`): conviene un blend equilibrado
- adopción tardía (`3` o `4`): suelen pesar más variables estructurales como edad, raza, fee, estado, cantidad de fotos, rescuer y señales de descripción más completas

Por eso dejé un `region_aware_blend`.
No es la única opción, pero es una muy buena primera versión porque:

1. es interpretable
2. se puede ajustar rápido
3. permite testear tu intuición de negocio sin meter todavía una arquitectura muy compleja

Si después querés subir de nivel, los siguientes pasos naturales serían:

- hacer stacking con meta-modelo sobre probabilidades de tabular, texto e imagen
- transformar el problema en ordinal classification
- usar una compuerta aprendida que decida cuánto pesar cada submodelo según el caso